# Cross-Package Comparison — polars_reg v0.2.0

This notebook demonstrates `compare()`, which runs the same regression across multiple packages and shows a side-by-side comparison of coefficients and standard errors.

**Backends supported:**
- **pyfixest** — OLS, FE, IV, probit, logit, PPML, quantile
- **statsmodels** — OLS, WLS, probit, logit, PPML (GLM), quantile
- **linearmodels** — panel FE/RE/FD, IV (2SLS, LIML, GMM)
- **R** — fixest, plm, AER (via Rscript subprocess)
- **Stata** — reg, reghdfe, ivregress, xtreg, probit, logit (via batch mode subprocess)

> **Note on Stata:** pystata (Stata's Python API) requires the native shared library, which is not directly accessible from WSL2. The `compare()` function uses Stata's batch mode via subprocess instead, which works seamlessly from WSL2.

## Imports

In [ ]:
import numpy as np
import polars as pl
import polars_reg as pr

## Data Simulation

We generate a panel dataset with known parameters so we can verify that all packages recover the true values.

| Parameter | True value |
|-----------|------------|
| β₁ (x1) | 2.0 |
| β₂ (x2) | -0.5 |
| β₃ (x_endog) | 0.8 |
| Intercept | 1.0 |
| Entity FE | N(0, 0.5²) across 50 firms |
| Error | N(0, 0.5²) |

In [ ]:
rng = np.random.default_rng(42)
n_firms, n_years = 50, 20
n = n_firms * n_years

# Panel structure
firm_id = np.repeat(np.arange(n_firms), n_years)
year_id = np.tile(np.arange(2000, 2000 + n_years), n_firms)

# Entity fixed effects
fe = rng.standard_normal(n_firms) * 0.5

# Regressors
x1 = rng.standard_normal(n)
x2 = rng.standard_normal(n)
z1 = rng.standard_normal(n)
z2 = rng.standard_normal(n)

# Endogenous variable (correlated with error via u)
u = rng.standard_normal(n) * 0.5
x_endog = 0.5 * z1 + 0.3 * z2 + 0.4 * u

# Outcome
y = 1.0 + 2.0 * x1 - 0.5 * x2 + 0.8 * x_endog + fe[firm_id] + u

# Binary outcome for probit/logit
prob = 1.0 / (1.0 + np.exp(-(0.5 + 1.0 * x1 - 0.3 * x2)))
y_binary = (rng.uniform(size=n) < prob).astype(float)

# Count outcome for PPML
y_count = rng.poisson(np.exp(0.5 + 0.3 * x1 + 0.2 * x2)).astype(float)

df = pl.DataFrame({
    "y": y, "x1": x1, "x2": x2,
    "x_endog": x_endog, "z1": z1, "z2": z2,
    "y_binary": y_binary, "y_count": y_count,
    "firm_id": firm_id, "year_id": year_id,
})
print(f"Panel: {n_firms} firms × {n_years} years = {n} observations")
df.head(5)

---

## 1. OLS — All Backends

Run plain OLS with robust (HC1) standard errors across every available package:

In [ ]:
report = pr.compare("ols", "y ~ x1 + x2", df, vcov="HC1")
print(report.summary())

## 2. OLS + Fixed Effects + Clustered SEs

Absorbed firm fixed effects with cluster-robust SEs — the reghdfe use case:

In [ ]:
report = pr.compare(
    "ols", "y ~ x1 + x2 | firm_id", df,
    cluster=["firm_id"],
    rtol=1e-4,
)
print(report.summary())

## 3. IV / 2SLS

Instrumental variables with an endogenous regressor:

In [ ]:
report = pr.compare(
    "iv2sls", "y ~ x1 + x2 || x_endog ~ z1 + z2", df,
    backend=["pyfixest", "linearmodels", "r"],
    rtol=1e-3,
)
print(report.summary())

## 4. Panel Fixed Effects

Within estimator — compared against linearmodels PanelOLS and R's plm:

In [ ]:
report = pr.compare(
    "panel_fe", "y ~ x1 + x2", df,
    entity="firm_id", time="year_id",
    backend=["pyfixest", "linearmodels", "r"],
    rtol=5e-2,
)
print(report.summary())

## 5. Probit

In [ ]:
report = pr.compare(
    "probit", "y_binary ~ x1 + x2", df,
    backend=["pyfixest", "statsmodels"],
    rtol=1e-3,
)
print(report.summary())

## 6. Logit

In [ ]:
report = pr.compare(
    "logit", "y_binary ~ x1 + x2", df,
    backend=["pyfixest", "statsmodels"],
    rtol=1e-3,
)
print(report.summary())

## 7. PPML (Poisson Pseudo-Maximum Likelihood)

In [ ]:
report = pr.compare(
    "ppml", "y_count ~ x1 + x2", df,
    backend=["pyfixest", "statsmodels"],
    rtol=1e-3,
)
print(report.summary())

## 8. Quantile Regression (Median)

In [ ]:
report = pr.compare(
    "quantreg", "y ~ x1 + x2", df,
    tau=0.5,
    backend=["statsmodels"],
    rtol=1e-2,
)
print(report.summary())

---

## 9. regtable() — Great Tables Rendering

`regtable()` now returns a `great_tables.GT` object. It renders natively in Jupyter and supports `.as_latex()` for papers:

In [ ]:
r1 = pr.ols("y ~ x1 + x2", data=df)
r2 = pr.ols("y ~ x1 + x2", data=df, vcov="HC1")
r3 = pr.ols("y ~ x1 + x2 | firm_id", data=df, cluster=["firm_id"])

# Renders as a styled HTML table in Jupyter
pr.regtable(r1, r2, r3, labels=["OLS", "Robust", "FE+Cluster"])

In [ ]:
# Add a title via GT's fluent API
pr.regtable(r1, r2, r3, labels=["OLS", "Robust", "FE+Cluster"]).tab_header(
    title="Table 1: OLS Results",
    subtitle="Dependent variable: y"
)

In [ ]:
# LaTeX export for papers
print(pr.regtable(r1, r2, r3, labels=["OLS", "Robust", "FE+Cluster"]).as_latex())

---

## 10. Inspecting Backend Code

Each backend result includes the equivalent code that was run, useful for reproducibility:

In [ ]:
report = pr.compare("ols", "y ~ x1 + x2", df, vcov="HC1", backend=["pyfixest", "statsmodels", "r"])
for name, result in report.backends.items():
    print(f"--- {name} ---")
    print(result.code)
    print()

# Also available: to_stata() and to_r() for manual verification
print("--- Stata (code only) ---")
print(pr.to_stata("ols", "y ~ x1 + x2", vcov="HC1"))
print()
print("--- R (code only) ---")
print(pr.to_r("ols", "y ~ x1 + x2", vcov="HC1"))